# Step 7 — Conformal Prediction (C3), and the C1-via-C3 test

**RESS 2025 — GAN-Conformal-RUL**

Split conformal prediction gives distribution-free, coverage-guaranteed RUL intervals. This
notebook does two things:

1. **Establishes C3 on the real-data baseline** — per-stage PICP (coverage) and MPIW (width) at
   several confidence levels, multi-seed. The overall coverage *must* hold (~1−α) by construction;
   the real question is whether **near-failure** coverage holds and at what width.
2. **Tests C1's true claim** — does GAN augmentation of *training* narrow the intervals at matched
   coverage? Point RMSE was ~flat (Step 6b); interval width can move independently. This is the
   honest test of the synergy hypothesis.

**Guards baked in:** calibration is real-only (asserted); the quantile uses the finite-sample
correction ⌈(n+1)(1−α)⌉/n; every comparison is multi-seed mean ± std; MPIW is always read next to
PICP so a narrower-but-under-covering interval is never mistaken for an improvement.

## 1. Setup

In [ ]:
import os, sys, shutil
os.chdir('/content')
REPO_PATH = '/content/RESS_2025_GAN_Conformal_RUL'
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)
!git clone https://github.com/f-khadija-benzine/RESS_2025_GAN_Conformal_RUL.git {REPO_PATH}
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH); sys.path.insert(0, f'{REPO_PATH}/src')
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = f'{REPO_PATH}/figures'; os.makedirs(SAVE_DIR, exist_ok=True)
CKPT_DIR = '/content/drive/MyDrive/ress_checkpoints'
import torch
print('CUDA:', torch.cuda.is_available())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

from data_loader import XJTUSYLoader
from health_indicator_v3 import HealthIndicatorPipeline
from windowing import prepare_all_folds, build_folds, WINDOW_SIZE
from model import ModelConfig, RULTrainer
from gan import StageGAN, GANConfig
from conformal import (calibrate_and_evaluate, average_conformal,
                       print_conformal, assert_real_calibration)

TARGET = 2
SEEDS = [1, 2, 3]                 # multi-seed per the consistency contract
ALPHAS = [0.20, 0.15, 0.10, 0.05]  # 80 / 85 / 90 / 95% intervals

## 2. Data (piecewise, log_clip) + GANs

In [ ]:
CANDIDATES = ['/content/drive/MyDrive/XJTU-SY',
              '/content/drive/MyDrive/XJTU-SY_Bearing_Datasets',
              '/content/drive/MyDrive/data/XJTU-SY']
DATA_ROOT = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert DATA_ROOT
all_data = XJTUSYLoader(DATA_ROOT).load_all()
pipeline = HealthIndicatorPipeline(fpt_consecutive=5, fpt_min_relative_rise=0.20)
results = pipeline.process_all(all_data, verbose=False)
fold_data = prepare_all_folds(results, scaling_method='log_clip',
                              rul_target='piecewise', verbose=False)
folds = build_folds(results)

# record the real calibration sizes NOW, to guard against later leaks
CAL_SIZES = {d['fold']: len(d['y_rul_cal']) for d in fold_data}
print('real calibration sizes per fold:', CAL_SIZES)

gans = {}
for k in range(1, 6):
    ck = torch.load(f'{CKPT_DIR}/gan_fold{k}.pt',
                    map_location='cuda' if torch.cuda.is_available() else 'cpu')
    g = StageGAN(GANConfig(**ck['config']))
    g.G.load_state_dict(ck['generator']); g.D.load_state_dict(ck['critic'])
    gans[k] = g
print(f'{len(gans)}/5 GANs loaded.')

## 3. Helpers — augmentation (training only) and a full conformal pass

In [ ]:
def augment_fold(gan, d, ratio, seed=0):
    """Add synthetic S3 windows to TRAINING only. Labels sampled from real S3
    RUL (the method that removed the kNN skew in Step 6b)."""
    rng = np.random.default_rng(seed)
    Xtr, ytr, s = d['X_train'], d['y_rul_train'], d['y_stage_train']
    s3 = s == TARGET
    n_real = int(s3.sum()); n_syn = int(round(ratio * n_real))
    if n_syn == 0:
        return Xtr, ytr, s
    X_syn = gan.sample(n_syn, TARGET)
    y_syn = rng.choice(ytr[s3], size=n_syn, replace=True)
    st_syn = np.full(n_syn, TARGET, dtype=s.dtype)
    return (np.concatenate([Xtr, X_syn.astype(np.float32)]),
            np.concatenate([ytr, y_syn.astype(np.float32)]),
            np.concatenate([s, st_syn]))


def one_pass(d, cfg, seed, augment_ratio=0.0):
    """Train (optionally augmented), then run conformal at all ALPHAS.
    Returns {alpha: conformal_result_dict}. Calibration stays REAL."""
    k = d['fold']
    if augment_ratio > 0:
        Xtr, ytr, str_ = augment_fold(gans[k], d, augment_ratio, seed=seed)
    else:
        Xtr, ytr, str_ = d['X_train'], d['y_rul_train'], d['y_stage_train']

    cfg = ModelConfig(**{**cfg.__dict__, 'seed': seed})
    tr = RULTrainer(cfg)
    tr.fit(Xtr, ytr, d['X_val'], d['y_rul_val'],
           stage_train=str_, stage_val=d['y_stage_val'],
           mask_healthy=False, verbose=False)

    # GUARD: calibration must be the untouched real calibration set
    assert_real_calibration(CAL_SIZES[k], d['y_rul_cal'])

    y_cal_pred = tr.predict(d['X_cal'])
    y_test_pred = tr.predict(d['X_test'])
    return {a: calibrate_and_evaluate(
                d['y_rul_cal'], y_cal_pred,
                d['y_rul_test'], y_test_pred, d['y_stage_test'], alpha=a)
            for a in ALPHAS}

## 4. Run baseline and +GAN, multi-seed

For each seed: train all 5 folds, conformalize, average across folds. Then aggregate seeds into
mean ± std. ~2 configs × 3 seeds × 5 folds trainings; budget accordingly.

In [ ]:
base_cfg = ModelConfig(epochs=100, patience=25, lr=5e-4)
AUG_RATIO = 0.75      # best setting from the Step 6b sweep

def run_config(augment_ratio):
    """Returns {alpha: {seed: fold-averaged conformal}} across seeds."""
    per_seed = {a: [] for a in ALPHAS}
    for seed in SEEDS:
        fold_res = {a: [] for a in ALPHAS}
        for d in fold_data:
            passes = one_pass(d, base_cfg, seed, augment_ratio)
            for a in ALPHAS:
                fold_res[a].append(passes[a])
        for a in ALPHAS:
            per_seed[a].append(average_conformal(fold_res[a]))
        print(f"  seed {seed} done")
    return per_seed

print('BASELINE (real only)…')
base_seeds = run_config(augment_ratio=0.0)
print(f'+GAN (ratio {AUG_RATIO})…')
gan_seeds = run_config(augment_ratio=AUG_RATIO)

## 5. Aggregate seeds → mean ± std, per stage, per alpha

In [ ]:
def agg(per_seed, subset, metric):
    """mean, std across seeds for one subset/metric, per alpha."""
    out = {}
    for a in ALPHAS:
        vals = [s[subset][metric] for s in per_seed[a]]
        out[a] = (float(np.mean(vals)), float(np.std(vals)))
    return out

print('C3 BASELINE — coverage should hold; near-failure is the row to watch\n')
for a in ALPHAS:
    print(f"--- target {1-a:.0%} (alpha={a}) ---")
    print(f"  {'subset':10s} {'PICP (mean±std)':>20s} {'MPIW (mean±std)':>20s}")
    for sub in ['healthy','early','nearfail','postFPT','overall']:
        pm = agg(base_seeds, sub, 'picp')[a]
        wm = agg(base_seeds, sub, 'mpiw')[a]
        print(f"  {sub:10s} {pm[0]:8.3f} ± {pm[1]:.3f}      {wm[0]:8.4f} ± {wm[1]:.4f}")
    print()

## 6. The C1-via-C3 test: does augmentation narrow intervals at matched coverage?

For each stage and alpha: compare baseline vs +GAN on **both** PICP and MPIW. A width reduction
only counts if coverage is preserved (PICP not lower). And the MPIW gap must exceed the seed std,
or it is within noise — a legitimate negative, exactly as Step 6b taught us.

In [ ]:
print('C1-via-C3: near-failure and overall, baseline vs +GAN\n')
for a in ALPHAS:
    print(f"=== target {1-a:.0%} ===")
    for sub in ['nearfail', 'overall']:
        bp, bps = agg(base_seeds, sub, 'picp')[a]
        bw, bws = agg(base_seeds, sub, 'mpiw')[a]
        gp, gps = agg(gan_seeds,  sub, 'picp')[a]
        gw, gws = agg(gan_seeds,  sub, 'mpiw')[a]
        dw = gw - bw
        noise = np.hypot(bws, gws)      # combined seed std
        # verdict: narrower AND coverage preserved AND gap beyond noise
        if gp < bp - 0.02:
            verdict = 'GAN under-covers — width not comparable'
        elif dw < -noise:
            verdict = f'GAN NARROWER by {-dw:.4f} (> noise {noise:.4f}) — C1 helps'
        elif dw > noise:
            verdict = f'GAN WIDER by {dw:.4f} — C1 hurts'
        else:
            verdict = f'within noise ({noise:.4f}) — no effect'
        print(f"  {sub:9s} PICP {bp:.3f}->{gp:.3f} | MPIW {bw:.4f}->{gw:.4f} | {verdict}")
    print()

## 7. Reliability plot (baseline) — coverage vs target

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
targets = [1 - a for a in ALPHAS]
for sub, color in [('overall','#2166d4'), ('nearfail','#e34948')]:
    picp = [agg(base_seeds, sub, 'picp')[a][0] for a in ALPHAS]
    err = [agg(base_seeds, sub, 'picp')[a][1] for a in ALPHAS]
    ax.errorbar(targets, picp, yerr=err, marker='o', label=sub, color=color, capsize=3)
ax.plot([0.75,1.0],[0.75,1.0], ls='--', color='gray', label='ideal')
ax.set_xlabel('target coverage (1−α)'); ax.set_ylabel('empirical PICP')
ax.set_title('Conformal reliability — baseline')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/07_reliability_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

## Reading this

- **Overall PICP ≈ target** confirms the conformal implementation is correct (guaranteed by
  construction on exchangeable data). If it's far off, suspect a calibration leak or a broken split.
- **near-failure PICP < target** would mean coverage fails exactly where it matters — a real,
  reportable finding about the limits of exchangeability in the degraded regime.
- **C1 verdict:** if +GAN narrows near-failure MPIW beyond the seed noise *without* dropping PICP,
  C1 stands on its true (conformal) claim. If within noise, C1 is a controlled negative and the
  paper rests on C2 (multi-task) and C3 (calibrated intervals) — a legitimate outcome.